# 03 — Spatial Filtering via Common Spatial Patterns (CSP)

**Goal:** formulate CSP from scratch (NumPy/SciPy, matching Section 4.2-4.3's generalized-eigenvalue derivation), then cross-verify against `mne.decoding.CSP`.

> **Single-subject scratch space only.** This notebook fits CSP on one subject's trials for illustration. LOSO-CV subject-splitting happens in Notebook 04 — nothing here should be reused as a leakage-free accuracy estimate.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from mne.decoding import CSP as MNE_CSP

from src.data_loader import get_dataset, get_paradigm
from src.feature_extraction import CustomCSP, trace_normalized_covariance

## 1. Load one subject's epochs as arrays

In [ ]:
dataset = get_dataset()
paradigm = get_paradigm()
subject_id = dataset.subject_list[0]

X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[subject_id], return_epochs=False)
unique_labels = sorted(set(labels))
y = np.array([0 if l == unique_labels[0] else 1 for l in labels])
print(f'X shape: {X.shape}  (trials, channels, times)')
print(f'Class balance: {np.bincount(y)}')

## 2. Trace-normalized class covariances (Section 4.2)

In [ ]:
X1, X2 = X[y == 0], X[y == 1]

C1 = np.mean([trace_normalized_covariance(trial) for trial in X1], axis=0)
C2 = np.mean([trace_normalized_covariance(trial) for trial in X2], axis=0)
print('C1 shape:', C1.shape, ' trace:', np.trace(C1))
print('C2 shape:', C2.shape, ' trace:', np.trace(C2))

## 3. Solve the generalized eigenvalue problem

In [ ]:
eigenvalues, eigenvectors = eigh(C1, C1 + C2)
order = np.argsort(eigenvalues)[::-1]
eigenvalues_sorted = eigenvalues[order]

plt.figure(figsize=(7, 3))
plt.plot(eigenvalues_sorted, 'o-')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.title('Sorted generalized eigenvalues (near 1 -> Class 1, near 0 -> Class 2)')
plt.xlabel('component index')
plt.ylabel('eigenvalue')
plt.show()

## 4. Fit `CustomCSP`, transform to log-variance features, and visualize separation

In [ ]:
custom_csp = CustomCSP(n_components=4)
features = custom_csp.fit_transform(X, y)

fig, ax = plt.subplots(figsize=(6, 5))
for cls, color, name in [(0, 'tab:blue', 'left_hand'), (1, 'tab:orange', 'right_hand')]:
    mask = y == cls
    ax.scatter(features[mask, 0], features[mask, -1], label=name, alpha=0.7, color=color)
ax.set_xlabel('log-var, component 1 (most class-1-like)')
ax.set_ylabel('log-var, component 4 (most class-2-like)')
ax.set_title(f'CSP log-variance feature separation — Subject {subject_id}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Cross-verify against `mne.decoding.CSP`

In [ ]:
mne_csp = MNE_CSP(n_components=4, log=True, norm_trace=False)
mne_features = mne_csp.fit_transform(X, y)

# Filters can differ by sign/order (eigenvector sign ambiguity), so
# compare via correlation magnitude across matched components rather
# than a raw allclose.
corrs = [abs(np.corrcoef(features[:, i], mne_features[:, i])[0, 1]) for i in range(4)]
print('Per-component |correlation| between CustomCSP and mne.decoding.CSP features:')
for i, c in enumerate(corrs):
    print(f'  component {i}: {c:.3f}')
assert min(corrs) > 0.9, 'CustomCSP and mne CSP features diverge more than expected'

## 6. CSP spatial patterns on scalp topography

In [ ]:
fig = mne_csp.plot_patterns(paradigm.get_data(dataset=dataset, subjects=[subject_id], return_epochs=True)[0].info,
                             ch_type='eeg', units='Patterns (AU)', size=1.5)
fig.suptitle('CSP spatial patterns — expect dipolar activation over sensorimotor cortex')
plt.show()

## Next
Proceed to **`04_Cross_Subject_Classification.ipynb`** for the full 50-subject LOSO-CV evaluation — CSP will be refit from scratch on every fold's training subjects there, never on pooled data.